# efa-kv-store Demo
Run this notebook on **node0** after starting the coordinator and servers.

```bash
# Terminal 1 (node0)
./build/coordinator

# Terminal 2-N (node1, node2, ...)
./build/server --coord node0

# Then launch jupyter on node0
jupyter notebook --no-browser --ip=0.0.0.0
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/efa-kv-store/build'))
import rdmastorage
print('rdmastorage loaded OK')

## 1. Connect
k and m are assigned automatically by the coordinator based on how many servers are alive.

In [ ]:
client = rdmastorage.Client('node0')
print(f'Connected: k={client.k}  m={client.m}  (tolerates {client.m} node failures)')

## 2. Basic put / get / delete

In [ ]:
client.put('hello', b'world')
val = client.get('hello')
print(f'get(hello) = {val}')
assert val == b'world'

In [ ]:
# Overwrite
client.put('hello', b'updated')
print(f'after overwrite: {client.get("hello")}')

In [ ]:
# Delete
client.delete('hello')
try:
    client.get('hello')
    print('ERROR: expected exception')
except RuntimeError as e:
    print(f'After delete: RuntimeError({e})  ✓')

## 3. Store and retrieve a file

In [ ]:
# Write a test file and store it
test_content = b'This is a test file stored via RDMA erasure coding!\n' * 100
with open('/tmp/test_input.txt', 'wb') as f:
    f.write(test_content)

with open('/tmp/test_input.txt', 'rb') as f:
    client.put('myfile', f.read())

print(f'Stored {len(test_content)} bytes under key "myfile"')

In [ ]:
# Retrieve and verify
data = client.get('myfile')
with open('/tmp/test_output.txt', 'wb') as f:
    f.write(data)

assert data == test_content, 'Data mismatch!'
print(f'Retrieved {len(data)} bytes  ✓')

## 4. Latency breakdown (per phase)

In [ ]:
import time

data_64k = bytes(range(256)) * 256  # 64 KB

# Warmup
for i in range(10):
    client.put(f'warmup-{i}', data_64k)

# Measure PUT phases
client.put('phase-test', data_64k)
enc, ctrl, rdma_w, commit = client.last_put_phases()
print(f'PUT 64KB phases:')
print(f'  encode:     {enc:.1f} us')
print(f'  ctrl RTT:   {ctrl:.1f} us  (request → slot grant)')
print(f'  RDMA write: {rdma_w:.1f} us  ← wire latency')
print(f'  commit RTT: {commit:.1f} us')
print(f'  total:      {enc+ctrl+rdma_w+commit:.1f} us')

In [ ]:
# Measure GET phases
client.get('phase-test')
dec, ctrl, rdma_r = client.last_get_phases()
print(f'GET 64KB phases:')
print(f'  ctrl RTT:   {ctrl:.1f} us  (request → slot info)')
print(f'  RDMA read:  {rdma_r:.1f} us  ← wire latency')
print(f'  decode:     {dec:.1f} us')
print(f'  total:      {ctrl+rdma_r+dec:.1f} us')

## 5. Throughput benchmark

In [ ]:
import statistics

def bench(op, obj_bytes, num_ops=200):
    data = (bytes(range(256)) * (obj_bytes // 256 + 1))[:obj_bytes]
    keys = [f'bench-{i:06d}' for i in range(num_ops)]

    # warmup
    for k in keys[:20]:
        client.put(k, data)
        if op == 'get':
            client.get(k)

    lats = []
    for k in keys:
        if op == 'put':
            t0 = time.perf_counter()
            client.put(k, data)
        else:
            client.put(k, data)
            t0 = time.perf_counter()
            client.get(k)
        lats.append((time.perf_counter() - t0) * 1e6)

    lats.sort()
    avg  = statistics.mean(lats)
    p50  = lats[int(len(lats) * 0.50)]
    p99  = lats[int(len(lats) * 0.99)]
    bw   = obj_bytes / (avg / 1e6) / 1e6
    print(f'{op.upper():3s} {obj_bytes//1024:5d}KB  avg={avg:7.1f}us  p50={p50:7.1f}us  p99={p99:7.1f}us  {bw:.1f} MB/s')

print(f'  op    size       avg        p50        p99       bw')
print('  ' + '-'*60)
for size in [4*1024, 16*1024, 64*1024, 256*1024]:
    bench('put', size)
    bench('get', size)
    print()